# 1\.1\.1 Load traffic counts

This notebook loads and stages NYC Automated Traffic Volume Count data for downstream multimodal mobility analysis\.

The workflow:
1\. Load the raw roadway traffic count CSV\.
2\. Inspect schema, missingness, temporal coverage, spatial coverage, directional coverage, and geometry availability\.
3\. Clean the traffic volume field, including comma\-formatted values such as \`1,137\`\.
4\. Create a timestamp from the year, month, day, hour, and minute fields\.
5\. Check for repeated segment\-direction\-timestamp records and aggregate them into a clean analysis grain\.
6\. Save the staged roadway traffic dataset as a parquet file for downstream notebooks\.

The final staged dataset is one row per roadway segment, travel direction, and timestamp, with traffic volume summed across repeated source measurements\.

## 1\.1\.1\.1 Load the traffic dataset

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# ---------------------------------------------------------------------
# File paths
# ---------------------------------------------------------------------

DATA_DIR = Path("source_data")
RAW_TRAFFIC_PATH = DATA_DIR / "Automated_Traffic_Volume_Counts_20260526.csv"


# ---------------------------------------------------------------------
# Load roadway traffic count data
# ---------------------------------------------------------------------

traffic_raw = pd.read_csv(
    RAW_TRAFFIC_PATH,
    low_memory=False
)

print("Raw traffic shape:", traffic_raw.shape)
traffic_raw.head()

Raw traffic shape: (271797, 14)


,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction
0,36906,Brooklyn,2023,11,15,20,45,20,42666,POINT (1008934.3110231468 170307.79075384818),AVENUE J,East 80 Street,East 81 Street,EB
1,36906,Brooklyn,2023,11,15,21,0,15,42666,POINT (1008934.3110231468 170307.79075384818),AVENUE J,East 80 Street,East 81 Street,EB
2,36906,Brooklyn,2023,11,15,21,15,26,42666,POINT (1008934.3110231468 170307.79075384818),AVENUE J,East 80 Street,East 81 Street,EB
3,36906,Brooklyn,2023,11,15,21,30,23,42666,POINT (1008934.3110231468 170307.79075384818),AVENUE J,East 80 Street,East 81 Street,EB
4,36906,Brooklyn,2023,11,15,21,45,13,42666,POINT (1008934.3110231468 170307.79075384818),AVENUE J,East 80 Street,East 81 Street,EB


## 1\.1\.1\.2 Dataset inspection

In [3]:
traffic_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271797 entries, 0 to 271796
Data columns (total 14 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   RequestID  271797 non-null  int64 
 1   Boro       271797 non-null  object
 2   Yr         271797 non-null  int64 
 3   M          271797 non-null  int64 
 4   D          271797 non-null  int64 
 5   HH         271797 non-null  int64 
 6   MM         271797 non-null  int64 
 7   Vol        271797 non-null  object
 8   SegmentID  271797 non-null  int64 
 9   WktGeom    271797 non-null  object
 10  street     271797 non-null  object
 11  fromSt     271797 non-null  object
 12  toSt       271797 non-null  object
 13  Direction  271797 non-null  object
dtypes: int64(7), object(7)
memory usage: 29.0+ MB


Datatypes all appear to be mostly correct\. Vol appears to have been brought in as a string, rather than an int\.

In [4]:
traffic_raw.isna().sum()

RequestID    0
Boro         0
Yr           0
M            0
D            0
HH           0
MM           0
Vol          0
SegmentID    0
WktGeom      0
street       0
fromSt       0
toSt         0
Direction    0
dtype: int64

Data is well populated \-\- there are no nulls\.

In [5]:
traffic_raw["Vol"].unique()[:20]

array(['20', '15', '26', '23', '13', '17', '14', '8', '11', '12', '3',
       '9', '5', '6', '1', '2', '4', '7', '28', '43'], dtype=object)

This looks like a straightforward transformation into ints\.

In [6]:
# Duplicate rows
print("Duplicates:", traffic_raw.duplicated().sum())
# Check for impossible / suspicious volume values


Duplicates: 0


Finding: No exact duplicate rows were found in the raw file

In [7]:
# Missing values
print(traffic_raw.isna().sum().sort_values(ascending=False))

RequestID    0
Boro         0
Yr           0
M            0
D            0
HH           0
MM           0
Vol          0
SegmentID    0
WktGeom      0
street       0
fromSt       0
toSt         0
Direction    0
dtype: int64


No nulls\! 

In [8]:
# Year coverage
print(traffic_raw["Yr"].value_counts().sort_index())

# Borough coverage
print(traffic_raw["Boro"].value_counts())

# Direction coverage
print(traffic_raw["Direction"].value_counts())

# Segment coverage
print(traffic_raw["SegmentID"].nunique())

# Temporal component ranges
print(traffic_raw[["Yr", "M", "D", "HH", "MM"]].describe())

# Geometry uniqueness / missingness
print(traffic_raw["WktGeom"].isna().sum(), traffic_raw["WktGeom"].nunique())

Yr
2023    85106
2024    95203
2025    82368
2026     9120
Name: count, dtype: int64
Boro
Brooklyn         101133
Queens            60794
Manhattan         56866
Bronx             45320
Staten Island      7684
Name: count, dtype: int64
Direction
NB    71017
EB    68147
WB    66317
SB    66316
Name: count, dtype: int64
322
                  Yr              M              D             HH  \
count  271797.000000  271797.000000  271797.000000  271797.000000   
mean     2024.057035       5.758643      14.529215      11.501433   
std         0.864373       3.363546       8.686172       6.921886   
min      2023.000000       1.000000       1.000000       0.000000   
25%      2023.000000       3.000000       7.000000       6.000000   
50%      2024.000000       5.000000      13.000000      12.000000   
75%      2025.000000       9.000000      22.000000      18.000000   
max      2026.000000      12.000000      31.000000      23.000000   

                  MM  
count  271797.000000  
mean    

Findings: The roadway traffic dataset contains approximately 272K traffic observations spanning 2023–2026, with strongest coverage concentrated in 2023–2025 and more limited early\-2026 measurements\. Spatial and directional coverage appeared well distributed across all five NYC boroughs, 322 roadway segments, and all four cardinal travel directions, while temporal inspection confirmed consistent quarter\-hour roadway traffic sampling and complete roadway geometry coverage\.

## 1\.1\.1\.3 Convert vol to numeric

In [9]:
traffic_staged = traffic_raw.copy()

# Convert traffic volume from string/object to numeric
traffic_staged["traffic_volume"] = pd.to_numeric(
    traffic_staged["Vol"],
    errors="coerce"
)


In [10]:
print(traffic_staged.isna().sum())

RequestID            0
Boro                 0
Yr                   0
M                    0
D                    0
HH                   0
MM                   0
Vol                  0
SegmentID            0
WktGeom              0
street               0
fromSt               0
toSt                 0
Direction            0
traffic_volume    3576
dtype: int64


In [11]:
invalid_vol_rows = traffic_raw[
    pd.to_numeric(traffic_raw["Vol"], errors="coerce").isna()
]

invalid_vol_rows.head()

,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction
2030,36190,Brooklyn,2023,6,14,19,45,"1,137",138751,POINT (1009836.9342472678 154801.48054019318),BELT PARKWAY,Dead End,Dead end,EB
2031,36190,Brooklyn,2023,6,14,20,0,"1,063",138751,POINT (1009836.9342472678 154801.48054019318),BELT PARKWAY,Dead End,Dead end,EB
2032,36190,Brooklyn,2023,6,14,20,15,"1,121",138751,POINT (1009836.9342472678 154801.48054019318),BELT PARKWAY,Dead End,Dead end,EB
2033,36190,Brooklyn,2023,6,14,20,30,"1,052",138751,POINT (1009836.9342472678 154801.48054019318),BELT PARKWAY,Dead End,Dead end,EB
2034,36190,Brooklyn,2023,6,14,20,45,"1,032",138751,POINT (1009836.9342472678 154801.48054019318),BELT PARKWAY,Dead End,Dead end,EB


Findings: Seems like we have “invalid” traffic volume values\. These appear to be normal traffic counts stored with comma separators \(for example, “1,137”\), so the issue was a formatting quirk rather than bad roadway measurement data\.

In [12]:
traffic_staged = traffic_raw.copy()

# Convert traffic volume from string/object to numeric
traffic_staged["traffic_volume"] = pd.to_numeric(
    traffic_staged["Vol"].str.replace(",", "", regex=False),
    errors="coerce"
)


In [13]:
print(traffic_staged.isna().sum())

RequestID         0
Boro              0
Yr                0
M                 0
D                 0
HH                0
MM                0
Vol               0
SegmentID         0
WktGeom           0
street            0
fromSt            0
toSt              0
Direction         0
traffic_volume    0
dtype: int64


Problem solved \-\- we no longer have any nulls for traffic\_volume

In [14]:
# Check for impossible / suspicious volume values
traffic_staged.query("traffic_volume < 0")

,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction,traffic_volume


Findings: No negative traffic volume values were identified in the staged dataset, suggesting that the roadway traffic counts appeared clean and internally consistent with no immediately suspicious volume measurements\.

### 1\.1\.1\.4 Date and timestamp conversion

In [15]:
# Create timestamp from date/time component fields
traffic_staged["timestamp"] = pd.to_datetime(
    {
        "year": traffic_staged["Yr"],
        "month": traffic_staged["M"],
        "day": traffic_staged["D"],
        "hour": traffic_staged["HH"],
        "minute": traffic_staged["MM"],
    },
    errors="coerce"
)

In [16]:
dupes = traffic_staged[
    traffic_staged.duplicated(
        subset=["SegmentID", "Direction", "timestamp"],
        keep=False
    )
].sort_values(
    ["SegmentID", "Direction", "timestamp"]
)

dupes.head(20)

,RequestID,Boro,Yr,M,D,HH,MM,Vol,SegmentID,WktGeom,street,fromSt,toSt,Direction,traffic_volume,timestamp
4917,36401,Brooklyn,2023,6,19,0,0,57,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,57,2023-06-19 00:00:00
26008,36401,Brooklyn,2023,6,19,0,0,18,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,18,2023-06-19 00:00:00
4918,36401,Brooklyn,2023,6,19,0,15,36,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,36,2023-06-19 00:15:00
26009,36401,Brooklyn,2023,6,19,0,15,34,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,34,2023-06-19 00:15:00
4919,36401,Brooklyn,2023,6,19,0,30,52,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,52,2023-06-19 00:30:00
26010,36401,Brooklyn,2023,6,19,0,30,14,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,14,2023-06-19 00:30:00
4920,36401,Brooklyn,2023,6,19,0,45,46,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,46,2023-06-19 00:45:00
26011,36401,Brooklyn,2023,6,19,0,45,22,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,22,2023-06-19 00:45:00
4921,36401,Brooklyn,2023,6,19,1,0,33,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,33,2023-06-19 01:00:00
26012,36401,Brooklyn,2023,6,19,1,0,24,50963,POINT (1022726.375963018 184975.11727740432),SUTTER AVENUE,South Conduit Avenue,Dead end,WB,24,2023-06-19 01:00:00


Findings: No exact duplicate rows were originally found in the raw file\. However, after creating timestamps, some segment\-direction\-timestamp combinations appeared more than once with different traffic counts, so these are repeated roadway measurements rather than simple duplicate rows\.

## 1\.1\.1\.5 Aggregate duplicated into a cleaner analysis frame

In [17]:
# ---------------------------------------------------------------------
# Aggregate repeated roadway traffic measurements to analysis grain
# ---------------------------------------------------------------------
# Some segment-direction-timestamp combinations appear more than once in
# the source file. These are repeated measurements for the same roadway
# interval, not exact duplicate rows. We sum traffic_volume so the staged
# table represents total observed volume for each segment, direction, and
# timestamp while preserving an observation_count audit field.

traffic_agg = (
    traffic_staged
    .groupby(
        ["SegmentID", "Direction", "timestamp"],
        as_index=False
    )
    .agg(
        traffic_volume=("traffic_volume", "sum"),
        Boro=("Boro", "first"),
        street=("street", "first"),
        fromSt=("fromSt", "first"),
        toSt=("toSt", "first"),
        WktGeom=("WktGeom", "first"),
        observation_count=("traffic_volume", "size")
    )
)

In [18]:
# Compare Shape and schema
print("Shape of traffic_staged:", traffic_staged.shape)
print("Shape of traffic_agg:", traffic_agg.shape)
print("\nSchema of traffic_agg:")
traffic_agg.info()

# Confirm aggregation grain is unique
print("Duplicates check:", traffic_agg.duplicated(
    subset=["SegmentID", "Direction", "timestamp"]
).sum())

# Confirm no missing values in key fields
print("No missing vals:", traffic_agg[
    ["SegmentID", "Direction", "timestamp", "traffic_volume"]
].isna().sum())

# Confirm timestamp coverage survived aggregation
print("Timestamp range:", traffic_agg["timestamp"].min(), traffic_agg["timestamp"].max())

traffic_agg.head()

Shape of traffic_staged: (271797, 16)
Shape of traffic_agg: (266651, 10)

Schema of traffic_agg:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266651 entries, 0 to 266650
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   SegmentID          266651 non-null  int64         
 1   Direction          266651 non-null  object        
 2   timestamp          266651 non-null  datetime64[ns]
 3   traffic_volume     266651 non-null  int64         
 4   Boro               266651 non-null  object        
 5   street             266651 non-null  object        
 6   fromSt             266651 non-null  object        
 7   toSt               266651 non-null  object        
 8   WktGeom            266651 non-null  object        
 9   observation_count  266651 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(6)
memory usage: 20.3+ MB
Duplicates check: 0
No missing vals: SegmentID    

,SegmentID,Direction,timestamp,traffic_volume,Boro,street,fromSt,toSt,WktGeom,observation_count
0,5164,WB,2023-09-06 00:00:00,16,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,POINT (932655.1292010084 168977.97581364735),1
1,5164,WB,2023-09-06 00:15:00,19,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,POINT (932655.1292010084 168977.97581364735),1
2,5164,WB,2023-09-06 00:30:00,10,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,POINT (932655.1292010084 168977.97581364735),1
3,5164,WB,2023-09-06 00:45:00,12,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,POINT (932655.1292010084 168977.97581364735),1
4,5164,WB,2023-09-06 01:00:00,16,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,POINT (932655.1292010084 168977.97581364735),1


Findings: Aggregation reduced the dataset from roughly 272K roadway traffic measurements to about 267K unique segment\-direction\-timestamp observations while preserving the full 2023–2026 temporal coverage window\. QA checks confirmed that the final staged traffic dataset no longer contained duplicate roadway interval records, missing key fields, or timestamp gaps, giving us a clean roadway traffic time series for downstream mobility analysis\.

### Rename the columns

In [19]:
traffic_agg = traffic_agg[
    [
        "SegmentID",
        "Boro",
        "street",
        "fromSt",
        "toSt",
        "Direction",
        "timestamp",
        "traffic_volume",
        "WktGeom",
        "observation_count",
    ]
].rename(
    columns={
        "SegmentID": "segment_id",
        "Boro": "borough",
        "street": "street_name",
        "fromSt": "from_street",
        "toSt": "to_street",
        "Direction": "direction",
        "WktGeom": "geometry_wkt",
    }
)

### Sort for downstream time\-series analysis

In [20]:
# Sort for downstream time-series analysis
traffic_agg = traffic_agg.sort_values(
    ["segment_id", "direction", "timestamp"]
).reset_index(drop=True)

traffic_agg.head()

,segment_id,borough,street_name,from_street,to_street,direction,timestamp,traffic_volume,geometry_wkt,observation_count
0,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:00:00,16,POINT (932655.1292010084 168977.97581364735),1
1,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:15:00,19,POINT (932655.1292010084 168977.97581364735),1
2,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:30:00,10,POINT (932655.1292010084 168977.97581364735),1
3,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 00:45:00,12,POINT (932655.1292010084 168977.97581364735),1
4,5164,Staten Island,GOETHALS ROAD NORTH,Staten Island Railway Line,Western Avenue,WB,2023-09-06 01:00:00,16,POINT (932655.1292010084 168977.97581364735),1


### 1\.1\.1\.6 Save staged traffic counts into Parquet file

In [21]:
# ---------------------------------------------------------------------
# Save staged roadway traffic dataset
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")
PIPELINE_DATA_DIR.mkdir(exist_ok=True)

TRAFFIC_PARQUET_PATH = (
    PIPELINE_DATA_DIR / "1.1.1.traffic_counts_staged.parquet"
)

traffic_agg.to_parquet(
    TRAFFIC_PARQUET_PATH,
    index=False
)

print(f"Saved staged traffic dataset to: {TRAFFIC_PARQUET_PATH}")

Saved staged traffic dataset to: pipeline_data/1.1.1.traffic_counts_staged.parquet


Close\. This parquet is now the staged roadway traffic dataset for downstream notebooks

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>